# COSC2753 Assignment 2 — Task 3: Gender & Usage Classification

Predict two targets from fashion item images:
- **gender** — who the item is intended for (Men, Women, Boys, Girls, Unisex)
- **usage** — what occasion it is suitable for (Casual, Formal, Sports, etc.)

Per the assignment spec, these may be treated as two separate classifiers **or** one combined class.
This notebook trains both approaches and compares them.

**Reused from Task 2:** split logic, dataset classes, model architectures (SmallCNN, ResNet18, MultiInputNet),
`fit()` / `run_epoch()` / `evaluate()` training utilities, metadata pipeline, EarlyStopping.

**New in Task 3:** combined-label approach, dual-head model, separate gender/usage evaluation.

## 0. Configuration — switch between TEST and REAL mode here

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  MODE SWITCH — set to True for a quick sanity-check run, False for full training
# ─────────────────────────────────────────────────────────────────────────────
TESTING_MODE = True   # <── change this only

if TESTING_MODE:
    # Fast smoke-test: small subset, large batches, few epochs
    SUBSET_SIZE   = 2000    # number of rows sampled from the full train CSV
    BATCH_SIZE    = 128     # larger batch = fewer optimizer steps per epoch
    EPOCHS        = 5       # just enough to confirm the forward pass works
    PATIENCE      = 3       # early-stopping patience
    NUM_WORKERS   = 0       # 0 avoids multiprocessing overhead on small runs
    IMG_SIZE      = (60, 80)  # smaller images → faster data loading
    LR            = 3e-4
    print("[TESTING MODE] Small subset, fast settings.")
else:
    # Full training run — use all data with production-quality settings
    SUBSET_SIZE   = None    # None = use entire dataset
    BATCH_SIZE    = 64
    EPOCHS        = 50
    PATIENCE      = 10
    NUM_WORKERS   = 4
    IMG_SIZE      = (80, 60)  # native image ratio
    LR            = 3e-4
    print("[REAL MODE] Full dataset, full training.")

## 1. Imports

In [ ]:
import os
import copy
import pickle
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)
import joblib

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import (
    resnet18, ResNet18_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights
)
from tqdm.auto import tqdm

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 2. Paths

In [ ]:
DATA_DIR         = Path("../data/raw/FashionDataset")
TRAIN_CSV        = DATA_DIR / "train" / "styles_train.csv"
IMAGES_TRAIN_DIR = DATA_DIR / "train" / "images_train"
TEST_PRED_CSV    = DATA_DIR / "test" / "styles_prediction.csv"
IMAGES_TEST_DIR  = DATA_DIR / "test" / "images_test"
PROCESSED_DIR    = Path("../data/processed")
OUTPUT_DIR       = Path("../outputs/task3_models")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_CSV, IMAGES_TRAIN_DIR, TEST_PRED_CSV, IMAGES_TEST_DIR]:
    print(f"[{'OK' if p.exists() else 'MISSING'}] {p}")

## 3. Load & clean dataset
*(Reused from Task 2 — same cleaning logic)*

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
df['id'] = df['id'].astype(str).str.strip()

# Keep only rows that have an image file
disk_ids = {p.stem for p in IMAGES_TRAIN_DIR.glob("*.jpg")}
missing  = set(df['id']) - disk_ids
df = df[~df['id'].isin(missing)].reset_index(drop=True)
print(f"Rows after image-alignment: {len(df)}")

# Optional subset for TESTING_MODE
if TESTING_MODE and SUBSET_SIZE:
    df = df.sample(SUBSET_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"[TESTING MODE] Subsampled to {len(df)} rows")

In [ ]:
# Duplicate-image groups (needed for a leak-free split)
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_to_group, group_ids = {}, []
for img_id in df['id']:
    h = file_hash(IMAGES_TRAIN_DIR / f"{img_id}.jpg")
    hash_to_group.setdefault(h, img_id)
    group_ids.append(hash_to_group[h])

df['dup_group'] = group_ids
print(f"{df['dup_group'].nunique()} unique image groups in {len(df)} rows")

## 4. Target label exploration

In [ ]:
print("=== gender ===")
print(df['gender'].value_counts())
print("\n=== usage ===")
print(df['usage'].value_counts())

In [ ]:
# Drop rows with missing target labels — these can't be trained on
before = len(df)
df = df.dropna(subset=['gender', 'usage']).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with missing gender or usage labels")

# Combined label for the joint-classifier approach
df['gender_usage'] = df['gender'] + '__' + df['usage']
print(f"\nUnique gender x usage combinations: {df['gender_usage'].nunique()}")
print(df['gender_usage'].value_counts().head(15))

In [ ]:
# Drop rare combined classes (< 5 samples) — can't split or train on them reliably
combo_counts = df['gender_usage'].value_counts()
rare_combos  = combo_counts[combo_counts < 5].index
if len(rare_combos):
    print(f"Dropping {df['gender_usage'].isin(rare_combos).sum()} rows with rare gender_usage combinations")
    df = df[~df['gender_usage'].isin(rare_combos)].reset_index(drop=True)

# Distribution plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df['gender'].value_counts().plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].set_title('Gender distribution'); axes[0].tick_params(axis='x', rotation=30)

df['usage'].value_counts().head(10).plot(kind='bar', ax=axes[1], color='#C44E52')
axes[1].set_title('Usage distribution (top 10)'); axes[1].tick_params(axis='x', rotation=30)

df['gender_usage'].value_counts().head(15).plot(kind='bar', ax=axes[2], color='#55A868')
axes[2].set_title('Combined label (top 15)'); axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Label encoding

In [ ]:
gender_enc       = LabelEncoder()
usage_enc        = LabelEncoder()
gender_usage_enc = LabelEncoder()

df['gender_label']       = gender_enc.fit_transform(df['gender'])
df['usage_label']        = usage_enc.fit_transform(df['usage'])
df['gender_usage_label'] = gender_usage_enc.fit_transform(df['gender_usage'])

N_GENDER       = len(gender_enc.classes_)
N_USAGE        = len(usage_enc.classes_)
N_COMBINED     = len(gender_usage_enc.classes_)

print(f"Gender classes ({N_GENDER}):        {list(gender_enc.classes_)}")
print(f"Usage classes ({N_USAGE}):          {list(usage_enc.classes_)}")
print(f"Combined classes ({N_COMBINED}):    {list(gender_usage_enc.classes_)[:8]} ...")

## 6. Train / Validation split
*(Same leak-free strategy as Tasks 1 & 2: StratifiedGroupKFold on masterCategory)*

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

master_counts     = df['masterCategory'].value_counts()
too_rare_master   = master_counts[master_counts < sgkf.get_n_splits()].index
df = df[~df['masterCategory'].isin(too_rare_master)].reset_index(drop=True)

train_idx, val_idx = next(sgkf.split(df, df['masterCategory'], groups=df['dup_group']))
train_data = df.iloc[train_idx].reset_index(drop=True)
val_data   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_data)}, Val: {len(val_data)}")
overlap = set(train_data['dup_group']) & set(val_data['dup_group'])
print(f"Duplicate groups in both splits: {len(overlap)} (should be 0)")

## 7. Metadata features
*(Same one-hot pipeline as Task 2 — reused directly)*

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Task 3 metadata: exclude gender and usage (they are the targets — leakage if included)
# Also exclude season (it was Task 2's target — still useful here as a feature)
META_COLS = ['masterCategory', 'subCategory', 'season']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
meta_train = ohe.fit_transform(train_data[META_COLS])
meta_val   = ohe.transform(val_data[META_COLS])
META_DIM   = meta_train.shape[1]

print(f"Metadata feature dimension: {META_DIM}")

joblib.dump(ohe, OUTPUT_DIR / 'ohe_metadata.joblib')
joblib.dump(gender_enc,       OUTPUT_DIR / 'gender_encoder.joblib')
joblib.dump(usage_enc,        OUTPUT_DIR / 'usage_encoder.joblib')
joblib.dump(gender_usage_enc, OUTPUT_DIR / 'gender_usage_encoder.joblib')
print("Encoders saved.")

## 8. Shared utilities
*(Reused verbatim from Task 2)*

In [ ]:
# ── Image transforms ──────────────────────────────────────────────────────────
H, W = IMG_SIZE

train_transform = T.Compose([
    T.Resize((H, W)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((H, W)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


# ── Weighted sampler for class imbalance ──────────────────────────────────────
def get_weighted_sampler(frame, label_col):
    counts  = frame[label_col].value_counts().to_dict()
    weights = frame[label_col].map(lambda c: 1.0 / counts[c]).values
    return WeightedRandomSampler(weights, len(weights), replacement=True)


# ── Dataset classes ───────────────────────────────────────────────────────────
class FashionImageDataset(Dataset):
    """Image-only dataset for single-label tasks."""
    def __init__(self, frame, image_dir, label_col, transform):
        self.frame     = frame.reset_index(drop=True)
        self.image_dir = image_dir
        self.label_col = label_col
        self.transform = transform

    def __len__(self): return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(self.image_dir / f"{row['id']}.jpg") as im:
            img = self.transform(im.convert('RGB'))
        return img, torch.tensor(row[self.label_col], dtype=torch.long)


class MultiInputDataset(Dataset):
    """Image + metadata dataset for single-label tasks."""
    def __init__(self, frame, metadata, labels, image_dir, transform):
        self.frame     = frame.reset_index(drop=True)
        self.metadata  = metadata
        self.labels    = labels
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self): return len(self.frame)

    def __getitem__(self, i):
        with Image.open(self.image_dir / f"{self.frame.iloc[i]['id']}.jpg") as im:
            img = self.transform(im.convert('RGB'))
        meta = torch.tensor(self.metadata[i], dtype=torch.float32)
        lbl  = torch.tensor(self.labels[i],   dtype=torch.long)
        return img, meta, lbl


class DualLabelDataset(Dataset):
    """Image + metadata dataset that returns TWO labels (gender, usage).
    Used by the dual-head model."""
    def __init__(self, frame, metadata, gender_labels, usage_labels, image_dir, transform):
        self.frame         = frame.reset_index(drop=True)
        self.metadata      = metadata
        self.gender_labels = gender_labels
        self.usage_labels  = usage_labels
        self.image_dir     = image_dir
        self.transform     = transform

    def __len__(self): return len(self.frame)

    def __getitem__(self, i):
        with Image.open(self.image_dir / f"{self.frame.iloc[i]['id']}.jpg") as im:
            img = self.transform(im.convert('RGB'))
        meta   = torch.tensor(self.metadata[i],      dtype=torch.float32)
        g_lbl  = torch.tensor(self.gender_labels[i], dtype=torch.long)
        u_lbl  = torch.tensor(self.usage_labels[i],  dtype=torch.long)
        return img, meta, g_lbl, u_lbl

In [ ]:
# ── Model building blocks (reused from Task 2) ────────────────────────────────

class ConvBlock(nn.Module):
    """Residual Conv Block with SiLU and BatchNorm."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.act   = nn.SiLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Sequential()
        )
    def forward(self, x):
        return self.act(self.bn2(self.conv2(self.act(self.bn1(self.conv1(x))))) + self.shortcut(x))


class ImprovedSmallImageEncoder(nn.Module):
    """Upgraded small CNN with residual connections (from Task 2)."""
    def __init__(self, out_dim=128, dropout=0.2):
        super().__init__()
        self.out_dim = out_dim
        self.stage1 = ConvBlock(3, 32);   self.pool1 = nn.MaxPool2d(2)
        self.stage2 = ConvBlock(32, 64);  self.pool2 = nn.MaxPool2d(2)
        self.stage3 = ConvBlock(64, 128); self.pool3 = nn.MaxPool2d(2)
        self.stage4 = ConvBlock(128, 256)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(256, out_dim),
            nn.BatchNorm1d(out_dim), nn.SiLU()
        )
    def forward(self, x):
        x = self.pool1(self.stage1(x))
        x = self.pool2(self.stage2(x))
        x = self.pool3(self.stage3(x))
        return self.proj(self.global_pool(self.stage4(x)).flatten(1))


class PretrainedResNet18Encoder(nn.Module):
    """ImageNet-pretrained ResNet18 as a feature extractor."""
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.out_dim  = out_dim
        self.proj     = nn.Linear(512, out_dim)
    def forward(self, x):
        return self.proj(self.backbone(x))


class MultiInputNet(nn.Module):
    """Image encoder + metadata MLP → single classification head."""
    def __init__(self, metadata_dim, n_classes, image_encoder):
        super().__init__()
        self.image = image_encoder
        self.meta  = nn.Sequential(
            nn.BatchNorm1d(metadata_dim),
            nn.Linear(metadata_dim, 128), nn.ReLU(), nn.Dropout(0.2)
        )
        self.head = nn.Sequential(
            nn.Linear(image_encoder.out_dim + 128, 256),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_classes)
        )
    def forward(self, image, metadata):
        return self.head(torch.cat([self.image(image), self.meta(metadata)], dim=1))


# ── NEW for Task 3: Dual-head model ───────────────────────────────────────────
class DualHeadNet(nn.Module):
    """Shared image+metadata backbone with two separate classification heads:
    one for gender, one for usage. Jointly trained with a weighted sum of losses."""
    def __init__(self, metadata_dim, n_gender, n_usage, image_encoder):
        super().__init__()
        self.image = image_encoder
        self.meta  = nn.Sequential(
            nn.BatchNorm1d(metadata_dim),
            nn.Linear(metadata_dim, 128), nn.ReLU(), nn.Dropout(0.2)
        )
        fused_dim = image_encoder.out_dim + 128
        self.shared = nn.Sequential(
            nn.Linear(fused_dim, 256), nn.ReLU(), nn.Dropout(0.3)
        )
        self.gender_head = nn.Linear(256, n_gender)
        self.usage_head  = nn.Linear(256, n_usage)

    def forward(self, image, metadata):
        fused  = torch.cat([self.image(image), self.meta(metadata)], dim=1)
        shared = self.shared(fused)
        return self.gender_head(shared), self.usage_head(shared)

In [ ]:
# ── Training utilities (reused from Task 2) ───────────────────────────────────

class EarlyStopping:
    def __init__(self, patience=5, delta=0, mode='min'):
        assert mode in ('max', 'min')
        self.patience = patience; self.delta = delta; self.mode = mode
        self.best_score = None; self.early_stop = False
        self.counter = 0; self.best_state = None

    def __call__(self, value, model):
        score = value if self.mode == 'max' else -value
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def load_best_model(self, model):
        model.load_state_dict(self.best_state)
        return model


def to_device(batch):
    """Move all tensors in batch to DEVICE; last tensor is always the label(s)."""
    *inputs, labels = batch
    return [x.to(DEVICE) for x in inputs], labels.to(DEVICE)


def run_epoch(model, loader, criterion, optimiser=None, desc=''):
    train_mode = optimiser is not None
    model.train() if train_mode else model.eval()
    running_loss, n, preds, actual = 0.0, 0, [], []
    with torch.set_grad_enabled(train_mode):
        for batch in tqdm(loader, desc=desc, leave=train_mode):
            inputs, labels = to_device(batch)
            logits = model(*inputs)
            loss   = criterion(logits, labels)
            if train_mode:
                optimiser.zero_grad(); loss.backward(); optimiser.step()
            running_loss += loss.item() * labels.size(0)
            n += labels.size(0)
            preds.extend(logits.argmax(1).detach().cpu().numpy())
            actual.extend(labels.cpu().numpy())
    return running_loss / n, f1_score(actual, preds, average='macro', zero_division=0)


def fit(model, loader_tr, loader_va, model_name='model', patience=PATIENCE, lr=LR):
    criterion    = nn.CrossEntropyLoss()
    optimiser    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler    = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=2)
    stopper      = EarlyStopping(patience=patience, mode='min')
    history      = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    best_val_f1  = None; best_epoch = None

    for epoch in range(EPOCHS):
        tag = f'{model_name} {epoch+1}/{EPOCHS}'
        tr_loss, tr_f1 = run_epoch(model, loader_tr, criterion, optimiser, desc=f'{tag} train')
        va_loss, va_f1 = run_epoch(model, loader_va, criterion, None,      desc=f'{tag} val')
        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
        history['train_f1'].append(tr_f1);    history['val_f1'].append(va_f1)
        current_lr = optimiser.param_groups[0]['lr']
        print(f'Epoch {epoch+1}/{EPOCHS} — tr_loss={tr_loss:.4f}  va_loss={va_loss:.4f}  '
              f'tr_f1={tr_f1:.4f}  va_f1={va_f1:.4f}  lr={current_lr:.2e}')
        scheduler.step(va_loss)
        stopper(va_loss, model)
        if stopper.counter == 0:
            best_val_f1 = va_f1; best_epoch = epoch + 1
        if stopper.early_stop:
            print(f'Early stopping at epoch {epoch+1} (best epoch={best_epoch})')
            break

    model = stopper.load_best_model(model)
    plot_training_curves(history, model_name)
    print(f'>>> {model_name}: best epoch={best_epoch}, val_f1={best_val_f1:.4f}')
    return model, best_val_f1, best_epoch


def evaluate(model, loader, label_col='single'):
    model.eval(); preds, actual = [], []
    with torch.no_grad():
        for batch in loader:
            inputs, labels = to_device(batch)
            preds.extend(model(*inputs).argmax(1).cpu().numpy())
            actual.extend(labels.cpu().numpy())
    return f1_score(actual, preds, average='macro', zero_division=0)


def plot_training_curves(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['train_f1'], label='Train'); axes[1].plot(history['val_f1'], label='Val')
    axes[1].set_title('Macro-F1'); axes[1].legend()
    plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()

## 9. Approach A — Two separate classifiers (gender | usage)

Train one model for each target independently. Simpler, but ignores the correlation between gender and usage.

In [ ]:
# ── Labels ────────────────────────────────────────────────────────────────────
y_gender_train = train_data['gender_label'].values
y_gender_val   = val_data['gender_label'].values
y_usage_train  = train_data['usage_label'].values
y_usage_val    = val_data['usage_label'].values

# ── Weighted samplers ─────────────────────────────────────────────────────────
sampler_gender = get_weighted_sampler(train_data, 'gender_label')
sampler_usage  = get_weighted_sampler(train_data, 'usage_label')

In [ ]:
# ─── 9.1 Baseline: Logistic Regression on metadata only ──────────────────────
BASELINE_DIR = OUTPUT_DIR / 'baselines'
BASELINE_DIR.mkdir(exist_ok=True)

logreg_gender = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
logreg_gender.fit(meta_train, y_gender_train)
logreg_gender_f1 = f1_score(y_gender_val, logreg_gender.predict(meta_val), average='macro', zero_division=0)

logreg_usage = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
logreg_usage.fit(meta_train, y_usage_train)
logreg_usage_f1 = f1_score(y_usage_val, logreg_usage.predict(meta_val), average='macro', zero_division=0)

print(f'Logistic Regression — gender val F1: {logreg_gender_f1:.4f}')
print(f'Logistic Regression — usage  val F1: {logreg_usage_f1:.4f}')

print('\nGender classification report:')
print(classification_report(y_gender_val, logreg_gender.predict(meta_val),
                             target_names=gender_enc.classes_, zero_division=0))
print('\nUsage classification report:')
print(classification_report(y_usage_val, logreg_usage.predict(meta_val),
                             target_names=usage_enc.classes_, zero_division=0))

joblib.dump(logreg_gender, BASELINE_DIR / 'logreg_gender.joblib')
joblib.dump(logreg_usage,  BASELINE_DIR / 'logreg_usage.joblib')

In [ ]:
# ─── 9.2 Gender classifier — MultiInput + ImprovedSmallCNN ───────────────────
train_ds_gender = MultiInputDataset(train_data, meta_train, y_gender_train, IMAGES_TRAIN_DIR, train_transform)
val_ds_gender   = MultiInputDataset(val_data,   meta_val,   y_gender_val,   IMAGES_TRAIN_DIR, eval_transform)

train_loader_gender = DataLoader(train_ds_gender, batch_size=BATCH_SIZE, sampler=sampler_gender,
                                  num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))
val_loader_gender   = DataLoader(val_ds_gender,   batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))

torch.manual_seed(RANDOM_STATE)
gender_model, gender_f1, gender_epoch = fit(
    MultiInputNet(META_DIM, N_GENDER, ImprovedSmallImageEncoder(out_dim=128)).to(DEVICE),
    train_loader_gender, val_loader_gender,
    model_name='MultiInput-ImprovedCNN (gender)'
)

torch.save(gender_model.state_dict(), OUTPUT_DIR / 'gender_multiinput_smallcnn.pt')
print(f'Saved gender model — val_f1={gender_f1:.4f}')

In [ ]:
# ─── 9.3 Gender classifier — MultiInput + Pretrained ResNet18 ────────────────
torch.manual_seed(RANDOM_STATE)
gender_resnet_model, gender_resnet_f1, gender_resnet_epoch = fit(
    MultiInputNet(META_DIM, N_GENDER, PretrainedResNet18Encoder(out_dim=128)).to(DEVICE),
    train_loader_gender, val_loader_gender,
    model_name='MultiInput-ResNet18Pretrained (gender)'
)
print(f'Pretrained ResNet18 gender — val_f1={gender_resnet_f1:.4f}')

In [ ]:
# ─── 9.4 Usage classifier — MultiInput + ImprovedSmallCNN ────────────────────
train_ds_usage = MultiInputDataset(train_data, meta_train, y_usage_train, IMAGES_TRAIN_DIR, train_transform)
val_ds_usage   = MultiInputDataset(val_data,   meta_val,   y_usage_val,   IMAGES_TRAIN_DIR, eval_transform)

train_loader_usage = DataLoader(train_ds_usage, batch_size=BATCH_SIZE, sampler=sampler_usage,
                                 num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))
val_loader_usage   = DataLoader(val_ds_usage,   batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))

torch.manual_seed(RANDOM_STATE)
usage_model, usage_f1, usage_epoch = fit(
    MultiInputNet(META_DIM, N_USAGE, ImprovedSmallImageEncoder(out_dim=128)).to(DEVICE),
    train_loader_usage, val_loader_usage,
    model_name='MultiInput-ImprovedCNN (usage)'
)

torch.save(usage_model.state_dict(), OUTPUT_DIR / 'usage_multiinput_smallcnn.pt')
print(f'Saved usage model — val_f1={usage_f1:.4f}')

In [ ]:
# ─── 9.5 Usage classifier — MultiInput + Pretrained ResNet18 ─────────────────
torch.manual_seed(RANDOM_STATE)
usage_resnet_model, usage_resnet_f1, usage_resnet_epoch = fit(
    MultiInputNet(META_DIM, N_USAGE, PretrainedResNet18Encoder(out_dim=128)).to(DEVICE),
    train_loader_usage, val_loader_usage,
    model_name='MultiInput-ResNet18Pretrained (usage)'
)
print(f'Pretrained ResNet18 usage — val_f1={usage_resnet_f1:.4f}')

## 10. Approach B — Single combined-label classifier (gender × usage)

Treats gender_usage as one combined class. Learns the joint distribution but has more classes and sparser per-class data.

In [ ]:
y_combined_train = train_data['gender_usage_label'].values
y_combined_val   = val_data['gender_usage_label'].values
sampler_combined = get_weighted_sampler(train_data, 'gender_usage_label')

train_ds_combined = MultiInputDataset(train_data, meta_train, y_combined_train, IMAGES_TRAIN_DIR, train_transform)
val_ds_combined   = MultiInputDataset(val_data,   meta_val,   y_combined_val,   IMAGES_TRAIN_DIR, eval_transform)

train_loader_combined = DataLoader(train_ds_combined, batch_size=BATCH_SIZE, sampler=sampler_combined,
                                    num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))
val_loader_combined   = DataLoader(val_ds_combined,   batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))

torch.manual_seed(RANDOM_STATE)
combined_model, combined_f1, combined_epoch = fit(
    MultiInputNet(META_DIM, N_COMBINED, ImprovedSmallImageEncoder(out_dim=128)).to(DEVICE),
    train_loader_combined, val_loader_combined,
    model_name='MultiInput-ImprovedCNN (gender x usage combined)'
)

torch.save(combined_model.state_dict(), OUTPUT_DIR / 'combined_gender_usage_smallcnn.pt')
print(f'Saved combined model — val_f1={combined_f1:.4f}')

## 11. Approach C — Dual-head model (shared backbone, two heads)

Trains one model that simultaneously predicts both gender and usage from a shared representation.
Loss = α × gender_loss + (1−α) × usage_loss.

In [ ]:
ALPHA = 0.5   # weight balance between gender and usage losses

train_ds_dual = DualLabelDataset(
    train_data, meta_train, y_gender_train, y_usage_train, IMAGES_TRAIN_DIR, train_transform)
val_ds_dual   = DualLabelDataset(
    val_data,   meta_val,   y_gender_val,   y_usage_val,   IMAGES_TRAIN_DIR, eval_transform)

train_loader_dual = DataLoader(train_ds_dual, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))
val_loader_dual   = DataLoader(val_ds_dual,   batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))


def fit_dual(model, loader_tr, loader_va, model_name='dual_model', patience=PATIENCE, lr=LR):
    """Training loop for the dual-head model. Returns gender F1, usage F1."""
    criterion  = nn.CrossEntropyLoss()
    optimiser  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=2)
    stopper    = EarlyStopping(patience=patience, mode='min')
    history    = {'train_loss': [], 'val_loss': [], 'train_f1_g': [], 'train_f1_u': [],
                  'val_f1_g':   [], 'val_f1_u': []}
    best_val_loss = None; best_epoch = None; best_val_f1_g = None; best_val_f1_u = None

    for epoch in range(EPOCHS):
        # ── Train ──
        model.train()
        tr_loss, tr_pg, tr_pu, tr_ag, tr_au = 0.0, [], [], [], []
        for img, meta, g_lbl, u_lbl in tqdm(loader_tr, desc=f'{model_name} {epoch+1}/{EPOCHS} train', leave=True):
            img, meta, g_lbl, u_lbl = img.to(DEVICE), meta.to(DEVICE), g_lbl.to(DEVICE), u_lbl.to(DEVICE)
            g_logit, u_logit = model(img, meta)
            loss = ALPHA * criterion(g_logit, g_lbl) + (1 - ALPHA) * criterion(u_logit, u_lbl)
            optimiser.zero_grad(); loss.backward(); optimiser.step()
            tr_loss += loss.item() * g_lbl.size(0)
            tr_pg.extend(g_logit.argmax(1).cpu().numpy()); tr_ag.extend(g_lbl.cpu().numpy())
            tr_pu.extend(u_logit.argmax(1).cpu().numpy()); tr_au.extend(u_lbl.cpu().numpy())
        tr_loss /= len(train_data)
        tr_f1_g = f1_score(tr_ag, tr_pg, average='macro', zero_division=0)
        tr_f1_u = f1_score(tr_au, tr_pu, average='macro', zero_division=0)

        # ── Validate ──
        model.eval()
        va_loss, va_pg, va_pu, va_ag, va_au = 0.0, [], [], [], []
        with torch.no_grad():
            for img, meta, g_lbl, u_lbl in loader_va:
                img, meta, g_lbl, u_lbl = img.to(DEVICE), meta.to(DEVICE), g_lbl.to(DEVICE), u_lbl.to(DEVICE)
                g_logit, u_logit = model(img, meta)
                loss = ALPHA * criterion(g_logit, g_lbl) + (1 - ALPHA) * criterion(u_logit, u_lbl)
                va_loss += loss.item() * g_lbl.size(0)
                va_pg.extend(g_logit.argmax(1).cpu().numpy()); va_ag.extend(g_lbl.cpu().numpy())
                va_pu.extend(u_logit.argmax(1).cpu().numpy()); va_au.extend(u_lbl.cpu().numpy())
        va_loss /= len(val_data)
        va_f1_g = f1_score(va_ag, va_pg, average='macro', zero_division=0)
        va_f1_u = f1_score(va_au, va_pu, average='macro', zero_division=0)

        for k, v in zip(['train_loss','val_loss','train_f1_g','train_f1_u','val_f1_g','val_f1_u'],
                         [tr_loss, va_loss, tr_f1_g, tr_f1_u, va_f1_g, va_f1_u]):
            history[k].append(v)

        print(f'Epoch {epoch+1}/{EPOCHS} — tr_loss={tr_loss:.4f}  va_loss={va_loss:.4f}  '
              f'tr_f1_g={tr_f1_g:.4f}  tr_f1_u={tr_f1_u:.4f}  '
              f'va_f1_g={va_f1_g:.4f}  va_f1_u={va_f1_u:.4f}')

        scheduler.step(va_loss)
        stopper(va_loss, model)
        if stopper.counter == 0:
            best_val_loss = va_loss; best_epoch = epoch + 1
            best_val_f1_g = va_f1_g; best_val_f1_u = va_f1_u
        if stopper.early_stop:
            print(f'Early stopping at epoch {epoch+1} (best={best_epoch})')
            break

    model = stopper.load_best_model(model)

    # Plot training curves
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(model_name, fontsize=13, fontweight='bold')
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['train_f1_g'], label='Train'); axes[1].plot(history['val_f1_g'], label='Val')
    axes[1].set_title('Gender Macro-F1'); axes[1].legend()
    axes[2].plot(history['train_f1_u'], label='Train'); axes[2].plot(history['val_f1_u'], label='Val')
    axes[2].set_title('Usage Macro-F1'); axes[2].legend()
    plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()

    print(f'>>> Best epoch={best_epoch}  val_f1_gender={best_val_f1_g:.4f}  val_f1_usage={best_val_f1_u:.4f}')
    return model, best_val_f1_g, best_val_f1_u, best_epoch


torch.manual_seed(RANDOM_STATE)
dual_model, dual_f1_g, dual_f1_u, dual_epoch = fit_dual(
    DualHeadNet(META_DIM, N_GENDER, N_USAGE, ImprovedSmallImageEncoder(out_dim=128)).to(DEVICE),
    train_loader_dual, val_loader_dual,
    model_name='DualHead-ImprovedCNN (gender + usage)'
)

torch.save(dual_model.state_dict(), OUTPUT_DIR / 'dual_head_gender_usage.pt')
print(f'Dual-head saved — gender_f1={dual_f1_g:.4f}  usage_f1={dual_f1_u:.4f}')

## 12. Evaluation & comparison across all approaches

In [ ]:
# ── Helper: evaluate dual-head on val set ────────────────────────────────────
def evaluate_dual(model, loader):
    model.eval(); pg, pu, ag, au = [], [], [], []
    with torch.no_grad():
        for img, meta, g_lbl, u_lbl in loader:
            img, meta = img.to(DEVICE), meta.to(DEVICE)
            g_logit, u_logit = model(img, meta)
            pg.extend(g_logit.argmax(1).cpu().numpy()); ag.extend(g_lbl.numpy())
            pu.extend(u_logit.argmax(1).cpu().numpy()); au.extend(u_lbl.numpy())
    f1_g = f1_score(ag, pg, average='macro', zero_division=0)
    f1_u = f1_score(au, pu, average='macro', zero_division=0)
    return f1_g, f1_u, pg, pu, ag, au


# ── Run evaluations ───────────────────────────────────────────────────────────
# Approach A
f1_g_cnn    = evaluate(gender_model,        val_loader_gender)
f1_g_resnet = evaluate(gender_resnet_model, val_loader_gender)
f1_u_cnn    = evaluate(usage_model,         val_loader_usage)
f1_u_resnet = evaluate(usage_resnet_model,  val_loader_usage)

# Approach B — decode combined label back to gender/usage
combined_model.eval(); cp, ca = [], []
with torch.no_grad():
    for batch in val_loader_combined:
        inputs, labels = to_device(batch)
        cp.extend(combined_model(*inputs).argmax(1).cpu().numpy())
        ca.extend(labels.cpu().numpy())

# Decode combined labels back to gender and usage
combo_classes   = gender_usage_enc.classes_
pred_gender_B   = [gender_enc.transform([c.split('__')[0]])[0] for c in gender_usage_enc.inverse_transform(cp)]
pred_usage_B    = [usage_enc.transform([c.split('__')[1]])[0]  for c in gender_usage_enc.inverse_transform(cp)]
true_gender_B   = val_data['gender_label'].values
true_usage_B    = val_data['usage_label'].values
f1_g_combined   = f1_score(true_gender_B, pred_gender_B, average='macro', zero_division=0)
f1_u_combined   = f1_score(true_usage_B,  pred_usage_B,  average='macro', zero_division=0)

# Approach C
f1_g_dual, f1_u_dual, _, _, _, _ = evaluate_dual(dual_model, val_loader_dual)

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'Model': [
        'LogReg (metadata baseline)',
        'A — MultiInput-SmallCNN (gender)',
        'A — MultiInput-ResNet18 (gender)',
        'A — MultiInput-SmallCNN (usage)',
        'A — MultiInput-ResNet18 (usage)',
        'B — Combined (gender decoded)',
        'B — Combined (usage decoded)',
        'C — DualHead (gender head)',
        'C — DualHead (usage head)',
    ],
    'Target': [
        'gender+usage', 'gender', 'gender', 'usage', 'usage',
        'gender', 'usage', 'gender', 'usage'
    ],
    'Val Macro-F1 (gender)': [
        logreg_gender_f1, f1_g_cnn, f1_g_resnet, None, None,
        f1_g_combined, None, f1_g_dual, None
    ],
    'Val Macro-F1 (usage)': [
        logreg_usage_f1, None, None, f1_u_cnn, f1_u_resnet,
        None, f1_u_combined, None, f1_u_dual
    ],
})

display(results_df.round(4))

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────────
gender_scores = {
    'LogReg baseline':    logreg_gender_f1,
    'A: SmallCNN':        f1_g_cnn,
    'A: ResNet18':        f1_g_resnet,
    'B: Combined':        f1_g_combined,
    'C: DualHead':        f1_g_dual,
}
usage_scores = {
    'LogReg baseline':    logreg_usage_f1,
    'A: SmallCNN':        f1_u_cnn,
    'A: ResNet18':        f1_u_resnet,
    'B: Combined':        f1_u_combined,
    'C: DualHead':        f1_u_dual,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scores, title in zip(axes,
                              [gender_scores, usage_scores],
                              ['Gender — Val Macro-F1', 'Usage — Val Macro-F1']):
    bars = ax.bar(scores.keys(), scores.values(), color=['#888'] + ['#4C72B0'] * 4)
    ax.set_ylim(0, 1); ax.set_title(title); ax.tick_params(axis='x', rotation=25)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices for best models ───────────────────────────────────────
# Determine best gender and usage model
best_gender_f1   = max(f1_g_cnn, f1_g_resnet, f1_g_combined, f1_g_dual)
best_gender_name = ['SmallCNN', 'ResNet18', 'Combined', 'DualHead'][
    [f1_g_cnn, f1_g_resnet, f1_g_combined, f1_g_dual].index(best_gender_f1)]
best_usage_f1    = max(f1_u_cnn, f1_u_resnet, f1_u_combined, f1_u_dual)
best_usage_name  = ['SmallCNN', 'ResNet18', 'Combined', 'DualHead'][
    [f1_u_cnn, f1_u_resnet, f1_u_combined, f1_u_dual].index(best_usage_f1)]

print(f'Best gender model: {best_gender_name} (val_f1={best_gender_f1:.4f})')
print(f'Best usage model:  {best_usage_name}  (val_f1={best_usage_f1:.4f})')

# Full classification reports for best models
# Re-collect predictions for the best models
def get_preds(model, loader):
    model.eval(); p, a = [], []
    with torch.no_grad():
        for batch in loader:
            inputs, labels = to_device(batch)
            p.extend(model(*inputs).argmax(1).cpu().numpy())
            a.extend(labels.cpu().numpy())
    return np.array(p), np.array(a)

best_g_model = {'SmallCNN': gender_model, 'ResNet18': gender_resnet_model,
                 'Combined': None,         'DualHead': None}[best_gender_name]
best_u_model = {'SmallCNN': usage_model,  'ResNet18': usage_resnet_model,
                 'Combined': None,         'DualHead': None}[best_usage_name]

# Gender confusion matrix
if best_g_model:
    g_pred, g_true = get_preds(best_g_model, val_loader_gender)
else:  # DualHead or Combined
    _, _, g_pred, _, g_true, _ = evaluate_dual(dual_model, val_loader_dual)
    g_pred = np.array(g_pred); g_true = np.array(g_true)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    g_true, g_pred, display_labels=gender_enc.classes_,
    ax=axes[0], xticks_rotation=30, colorbar=False, normalize='true')
axes[0].set_title(f'Gender — {best_gender_name} (normalised)')

# Usage confusion matrix
if best_u_model:
    u_pred, u_true = get_preds(best_u_model, val_loader_usage)
else:
    _, _, _, u_pred, _, u_true = evaluate_dual(dual_model, val_loader_dual)
    u_pred = np.array(u_pred); u_true = np.array(u_true)

ConfusionMatrixDisplay.from_predictions(
    u_true, u_pred, display_labels=usage_enc.classes_,
    ax=axes[1], xticks_rotation=30, colorbar=False, normalize='true')
axes[1].set_title(f'Usage — {best_usage_name} (normalised)')

plt.tight_layout()
plt.show()

print('\nGender classification report:')
print(classification_report(g_true, g_pred, target_names=gender_enc.classes_, zero_division=0))
print('\nUsage classification report:')
print(classification_report(u_true, u_pred, target_names=usage_enc.classes_,  zero_division=0))

## 13. Test-set prediction (submission)

In [ ]:
# Load the test prediction template
pred_df = pd.read_csv(TEST_PRED_CSV)
pred_df['id'] = pred_df['id'].astype(str).str.strip()
print(pred_df.shape)
pred_df.head(3)

In [ ]:
# ── Use the best gender and usage models for the final submission ──────────────
# Select whichever approach scored highest on validation
BEST_GENDER_MODEL = gender_model          # swap for gender_resnet_model / dual_model as needed
BEST_USAGE_MODEL  = usage_model           # swap for usage_resnet_model  / dual_model as needed

class TestDataset(Dataset):
    """Test-set dataset — image only (no labels). Returns (image, id)."""
    def __init__(self, ids, image_dir, transform):
        self.ids       = ids
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]
        path   = self.image_dir / f"{img_id}.jpg"
        with Image.open(path) as im:
            img = self.transform(im.convert('RGB'))
        return img, img_id


test_ids = pred_df['id'].tolist()
test_ds  = TestDataset(test_ids, IMAGES_TEST_DIR, eval_transform)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))


def predict_test_image_only(model, loader, encoder):
    """Run inference with an image-only model.
    Returns {id: decoded_label}."""
    model.eval(); results = {}
    with torch.no_grad():
        for imgs, ids in tqdm(loader, desc='Predicting test set'):
            logits = model(imgs.to(DEVICE))
            preds  = logits.argmax(1).cpu().numpy()
            for img_id, p in zip(ids, preds):
                results[img_id] = encoder.inverse_transform([p])[0]
    return results


def predict_test_multi(model, meta_features, loader_ids, encoder):
    """Run inference with a MultiInputNet model.
    meta_features: full OHE matrix aligned to loader_ids order.
    Returns {id: decoded_label}."""
    model.eval(); results = {}

    # Build metadata for test set
    test_meta_cols = pred_df[META_COLS] if all(c in pred_df.columns for c in META_COLS) else None
    if test_meta_cols is not None:
        test_meta = ohe.transform(test_meta_cols)
    else:
        print("WARNING: metadata columns not available in test CSV — using zero metadata")
        test_meta = np.zeros((len(pred_df), META_DIM))

    idx = 0
    with torch.no_grad():
        for imgs, ids in tqdm(loader, desc='Predicting test set (multi)'):
            bs   = imgs.size(0)
            meta = torch.tensor(test_meta[idx:idx+bs], dtype=torch.float32).to(DEVICE)
            logits = model(imgs.to(DEVICE), meta)
            preds  = logits.argmax(1).cpu().numpy()
            for img_id, p in zip(ids, preds):
                results[img_id] = encoder.inverse_transform([p])[0]
            idx += bs
    return results


# Predict gender and usage for the test set
gender_preds = predict_test_multi(BEST_GENDER_MODEL, None, test_ids, gender_enc)
usage_preds  = predict_test_multi(BEST_USAGE_MODEL,  None, test_ids, usage_enc)

pred_df['gender'] = pred_df['id'].map(gender_preds)
pred_df['usage']  = pred_df['id'].map(usage_preds)

pred_df.to_csv(OUTPUT_DIR / 'task3_predictions.csv', index=False)
print(f"Predictions saved — {len(pred_df)} rows")
pred_df.head()

## 14. Save models & encoders

In [ ]:
models_to_save = {
    'gender_multiinput_smallcnn.pt':       gender_model,
    'gender_multiinput_resnet18.pt':        gender_resnet_model,
    'usage_multiinput_smallcnn.pt':         usage_model,
    'usage_multiinput_resnet18.pt':         usage_resnet_model,
    'combined_gender_usage_smallcnn.pt':    combined_model,
    'dual_head_gender_usage.pt':            dual_model,
}

for fname, model in models_to_save.items():
    torch.save(model.state_dict(), OUTPUT_DIR / fname)
    print(f'Saved {fname}')

print(f'\nAll files in {OUTPUT_DIR}:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<45} {size_mb:.1f} MB')